# S4 - ML distribuido con Spark MLlib (Regresion)

**Actividad:** construir el notebook `04_ml_distribuido_regresion_practica.ipynb` sobre el entorno `lambda26` (`uso-pyspark`), entrenando y comparando modelos de regresion distribuida con Spark MLlib sobre un dataset real de sensores ambientales, y reportando metricas iniciales (RMSE, R2, MAE).


## 1. El dataset: sensores ambientales reales

`campo_electrico_particionado/` es la salida particionada en Parquet que construye Pre-S4 (Calidad de datos y particionamiento, segundo caso de uso) integrando tres fuentes reales de sensores (campo electrico, campo magnetico, variables ambientales), medidas minuto a minuto: 184 538 filas completas en las 9 variables, sin nulos, particionadas por mes (`AnioMes`).

El objetivo de hoy: estimar `Valor_CE` (campo electrico) a partir de las otras 8 variables medidas en el mismo instante — un problema de regresion multivariable clasico, sin ningun componente temporal (no se usa el minuto anterior ni el siguiente; cada fila es una observacion independiente). La version con horizonte de prediccion (estimar el valor del minuto **siguiente** usando historial) es contenido de S10 (Series de tiempo e inferencia en streaming), no de hoy.


## 2. Crear la `SparkSession`


In [7]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion4-ml-regresion")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark


In [8]:
ORIGEN_DATOS = "/opt/pre-s04-calidad-campo-electrico/artifacts/campo_electrico_particionado"
ARTIFACTS = "/opt/s04-ml-distribuido-regresion/artifacts"


## 3. Cargar el dataset (salida particionada de Pre-S4) y explorar las 9 variables

Se lee directo con `spark.read.parquet()` -- un Parquet particionado ya trae su propio esquema, no hace falta declararlo a mano como con un CSV (Pre-S4, paso 2). `AnioMes` reaparece como columna aunque no esta guardada dentro de ningun archivo: Spark la reconstruye a partir del nombre de la carpeta de particion (mismo *partition discovery* de S3).


In [9]:
VARIABLES_9 = [
    "Valor_CE", "Valor_CM", "TempOut", "OutHum",
    "WindSpeed", "Bar", "Rain", "SolarRad", "UVIndex",
]

df = spark.read.parquet(ORIGEN_DATOS)

df.printSchema()
print(f"Filas: {df.count():,}")
df.describe(VARIABLES_9).show()


root
 |-- FechaHora: timestamp (nullable = true)
 |-- Valor_CE: double (nullable = true)
 |-- Valor_CM: double (nullable = true)
 |-- TempOut: double (nullable = true)
 |-- OutHum: double (nullable = true)
 |-- WindSpeed: double (nullable = true)
 |-- Bar: double (nullable = true)
 |-- Rain: double (nullable = true)
 |-- SolarRad: double (nullable = true)
 |-- UVIndex: double (nullable = true)
 |-- AnioMes: string (nullable = true)



Filas: 184,538


26/09/02 01:01:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 4:===============>                                          (3 + 8) / 11]

+-------+-------------------+------------------+------------------+-----------------+-----------------+-----------------+--------------------+------------------+------------------+
|summary|           Valor_CE|          Valor_CM|           TempOut|           OutHum|        WindSpeed|              Bar|                Rain|          SolarRad|           UVIndex|
+-------+-------------------+------------------+------------------+-----------------+-----------------+-----------------+--------------------+------------------+------------------+
|  count|             184538|            184538|            184538|           184538|           184538|           184538|              184538|            184538|            184538|
|   mean|-0.9489558248165707|24278.279628585944|17.549393078931907|82.30233881368606|3.911056801309197|949.1927304945158|2.167575241955586...|174.62464641428866|1.2261539628694533|
| stddev| 0.8908501045924285|60.188433959341104| 3.008674476530922|7.323218566900627|4.87647261

## 4. Preparar el vector de predictores (`VectorAssembler`)

Spark MLlib no acepta columnas sueltas como entrada de un modelo — necesita una sola columna vectorial que agrupe todos los predictores. `VectorAssembler` hace exactamente eso: toma N columnas numericas y las combina en una columna `features` de tipo `Vector`. `Valor_CE` queda fuera de los predictores: es la columna objetivo (`label`), no un dato de entrada.


In [10]:
from pyspark.ml.feature import VectorAssembler

PREDICTORES = [v for v in VARIABLES_9 if v != "Valor_CE"]
print(f"Predictores ({len(PREDICTORES)}): {PREDICTORES}")

ensamblador = VectorAssembler(inputCols=PREDICTORES, outputCol="features")
df_ml = ensamblador.transform(df).select("features", "Valor_CE")

df_ml.show(5, truncate=False)


Predictores (8): ['Valor_CM', 'TempOut', 'OutHum', 'WindSpeed', 'Bar', 'Rain', 'SolarRad', 'UVIndex']
+--------------------------------------------+--------+
|features                                    |Valor_CE|
+--------------------------------------------+--------+
|[24327.1,20.9,77.0,11.3,949.6,0.0,349.0,2.0]|-2.17   |
|[24274.8,16.1,87.0,4.8,949.4,0.0,184.0,1.5] |-3.02   |
|[24250.4,19.6,83.0,4.8,952.0,0.0,411.0,2.1] |-0.27   |
|[24280.4,15.2,88.0,0.0,950.8,0.0,0.0,0.0]   |0.12    |
|[24278.1,13.8,84.0,3.2,950.8,0.0,0.0,0.0]   |-0.86   |
+--------------------------------------------+--------+
only showing top 5 rows


## 5. Dividir en entrenamiento y prueba

Division aleatoria simple (80/20), no cronologica — a diferencia de una tarea de pronostico (S10), aqui cada fila es una observacion independiente, sin orden temporal que preservar.


In [11]:
df_train, df_test = df_ml.randomSplit([0.8, 0.2], seed=42)

print(f"Entrenamiento: {df_train.count():,} filas")
print(f"Prueba: {df_test.count():,} filas")


Entrenamiento: 147,943 filas


[Stage 11:====================>                                    (4 + 7) / 11]

Prueba: 36,595 filas


## 6. Entrenar un modelo base: `LinearRegression`


In [12]:
from pyspark.ml.regression import LinearRegression

lr_base = LinearRegression(featuresCol="features", labelCol="Valor_CE")
modelo_base = lr_base.fit(df_train)

print("Coeficientes:", modelo_base.coefficients)
print("Intercepto:", modelo_base.intercept)


26/09/02 01:01:55 WARN Instrumentation: [4b317786] regParam is zero, which might cause numerical instability and overfitting.
netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory
                                                                                

Coeficientes: [-0.002788944930500952,0.145401072923687,0.006895259414812994,-0.07792759472452553,-0.04286953466350843,1.2398731396808425,-0.0007062868107990013,0.041606624500483594]
Intercepto: 104.71155302801812


## 7. Evaluar el modelo (RMSE, R2, MAE)


In [13]:
from pyspark.ml.evaluation import RegressionEvaluator

predicciones_base = modelo_base.transform(df_test)
predicciones_base.select("Valor_CE", "prediction").show(5)

def evaluar(predicciones, nombre):
    resultados = {}
    for metrica in ["rmse", "r2", "mae"]:
        evaluador = RegressionEvaluator(
            labelCol="Valor_CE", predictionCol="prediction", metricName=metrica
        )
        resultados[metrica.upper()] = evaluador.evaluate(predicciones)
    print(f"{nombre}: RMSE={resultados['RMSE']:.4f}  R2={resultados['R2']:.4f}  MAE={resultados['MAE']:.4f}")
    return resultados

resultados_base = evaluar(predicciones_base, "LinearRegression base")


+--------+--------------------+
|Valor_CE|          prediction|
+--------+--------------------+
|   -2.24|-0.19094105242808723|
|   -1.88|-0.16048159803641227|
|   -2.32| -0.2944109093496792|
|   -2.42| -0.3203480972033361|
|   -2.08|-0.01823941523674...|
+--------+--------------------+
only showing top 5 rows


[Stage 27:========================================================(11 + 0) / 11]

LinearRegression base: RMSE=0.7909  R2=0.2271  MAE=0.6087


## 8. Comparar configuraciones basicas (`regParam` / `elasticNetParam`)

El silabo pide comparar configuraciones basicas, no solo entrenar un unico modelo. `regParam` controla cuanto se penaliza la magnitud de los coeficientes (regularizacion); `elasticNetParam` mezcla penalizacion L1 (Lasso, `=1.0`) y L2 (Ridge, `=0.0`). Se prueban tres configuraciones simples, sin busqueda exhaustiva de hiperparametros (eso queda fuera del alcance de esta sesion):


In [14]:
configuraciones = [
    {"nombre": "Sin regularizacion", "regParam": 0.0, "elasticNetParam": 0.0},
    {"nombre": "Ridge (L2)", "regParam": 0.1, "elasticNetParam": 0.0},
    {"nombre": "Elastic Net (L1+L2)", "regParam": 0.1, "elasticNetParam": 0.5},
]

comparacion_configs = []
for config in configuraciones:
    lr = LinearRegression(
        featuresCol="features", labelCol="Valor_CE",
        regParam=config["regParam"], elasticNetParam=config["elasticNetParam"],
    )
    modelo = lr.fit(df_train)
    predicciones = modelo.transform(df_test)
    resultado = evaluar(predicciones, config["nombre"])
    resultado["Configuracion"] = config["nombre"]
    comparacion_configs.append(resultado)

import pandas as pd
pd.DataFrame(comparacion_configs)[["Configuracion", "RMSE", "R2", "MAE"]]


26/09/02 01:02:06 WARN Instrumentation: [b2fee45a] regParam is zero, which might cause numerical instability and overfitting.
                                                                                

Sin regularizacion: RMSE=0.7909  R2=0.2271  MAE=0.6087


Ridge (L2): RMSE=0.7962  R2=0.2166  MAE=0.6174


Elastic Net (L1+L2): RMSE=0.8091  R2=0.1910  MAE=0.6372


,Configuracion,RMSE,R2,MAE
0,Sin regularizacion,0.790852,0.227109,0.608656
1,Ridge (L2),0.796224,0.216575,0.617418
2,Elastic Net (L1+L2),0.809133,0.190964,0.637151


## 9. Comparar con un segundo algoritmo: `RandomForestRegressor`

`LinearRegression` asume una relacion lineal entre predictores y objetivo. `RandomForestRegressor` no — captura relaciones no lineales e interacciones entre variables sin necesitar ese supuesto. Comparar ambas familias (lineal vs. arboles) es la forma mas basica de saber si la relacion real es, de entrada, aproximadamente lineal.


In [15]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol="features", labelCol="Valor_CE",
    numTrees=50, maxDepth=8, seed=42,
)
modelo_rf = rf.fit(df_train)
predicciones_rf = modelo_rf.transform(df_test)

resultados_rf = evaluar(predicciones_rf, "Random Forest")


26/09/02 01:02:36 WARN DAGScheduler: Broadcasting large task binary with size 1029.6 KiB
26/09/02 01:02:38 WARN DAGScheduler: Broadcasting large task binary with size 1963.6 KiB
                                                                                

Random Forest: RMSE=0.6732  R2=0.4399  MAE=0.5001


## 9b. Importancia de variables: ?todas aportan?

`RandomForestRegressor` calcula, sin costo adicional, cuanto reduce cada variable el error del modelo en promedio, a lo largo de todos sus arboles -- a diferencia de los coeficientes de `LinearRegression` (3.6), que no son comparables entre si porque cada variable tiene una escala distinta (`Valor_CM` en miles, `Rain` entre 0 y 0.2). `featureImportances` si es directamente comparable: son proporciones que suman 1.0 entre todos los predictores.


In [18]:
importancias = list(zip(PREDICTORES, modelo_rf.featureImportances.toArray()))
importancias.sort(key=lambda x: x[1], reverse=True)

for variable, importancia in importancias:
    print(f"{variable:12s} {importancia:.4f}")


WindSpeed    0.3365
TempOut      0.2023
OutHum       0.1564
Valor_CM     0.1262
SolarRad     0.0800
Bar          0.0558
UVIndex      0.0428
Rain         0.0000


## 10. Comparacion final y seleccion


In [16]:
comparacion_final = pd.DataFrame(comparacion_configs + [
    {**resultados_rf, "Configuracion": "Random Forest"}
])[["Configuracion", "RMSE", "R2", "MAE"]]

comparacion_final.sort_values("RMSE")


,Configuracion,RMSE,R2,MAE
3,Random Forest,0.673222,0.439928,0.500095
0,Sin regularizacion,0.790852,0.227109,0.608656
1,Ridge (L2),0.796224,0.216575,0.617418
2,Elastic Net (L1+L2),0.809133,0.190964,0.637151


## 11. Guardar el modelo seleccionado

Elige, en base a la Tabla de la celda anterior, cual configuracion tuvo el mejor RMSE en tu propia corrida, y guardala. El nombre de variable `modelo_ganador` de abajo asume que fue el modelo de Random Forest — ajustalo segun tu resultado real.


In [17]:
modelo_ganador = modelo_rf  # ajusta esta linea segun tu propio resultado (celda anterior)

modelo_ganador.write().overwrite().save(f"{ARTIFACTS}/modelo_ce_regresion")
print(f"Modelo guardado en {ARTIFACTS}/modelo_ce_regresion")


Modelo guardado en /opt/s04-ml-distribuido-regresion/artifacts/modelo_ce_regresion


## 12. Documentar hallazgos y responder preguntas de reflexion

Agrega celdas markdown breves debajo de cada bloque de codigo (secciones 6-10) explicando que hiciste y que observaste — es la base directa de la evidencia tecnica que armaras en 4.3.1.

**Reflexion tecnica breve** (5 a 8 lineas): ?que diferencia de RMSE encontraste entre la configuracion sin regularizacion y la de Random Forest? ?por que `VectorAssembler` es un paso obligatorio en Spark MLlib y no en scikit-learn? ?que significaria un R2 cercano a 0 para este problema, y tu resultado se acerco a eso o se alejo?
